# SIA官方GSR月度附件

只解析已冻结公告明确链接的附件，不猜测URL，不拟合模型。每份附件的同比环比均来自同一版本；用Qlib日历/store/readback管理信息状态。PDF元数据只作为一致性证据，不能证明原始网页当时链接相同附件。

依赖版本见plan.json；环境补充命令：`uv pip install --python qlib_venv/bin/python pypdf==6.20.1 pypdfium2==5.14.0`。

In [1]:
from pathlib import Path
import os,json,hashlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,numpy as np,yaml
from qlib.data import D
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.sia_sales_pdf import collect_sales_pdfs
from src.data.qlib_native import build_qlib_store_from_frame
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/sia_sales_pdf_20261013';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/plan['price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='1min')).normalize().unique().sort_values()
objects,parent_audit=_source(root,plan['source'],['sources/manifest','sources/raw_documents'])
raw=root/plan['raw_cache'];raw.mkdir(parents=True,exist_ok=True)
from src.data.sia_sales_sources import article_metadata
from src.data.fed_capacity_vintages import _fetch
from datetime import datetime, timezone
supplement=raw/'supplementary_article.html';receipt=raw/'supplementary_article.receipt.json'
if not (supplement.exists() and receipt.exists()):
    payload=_fetch(plan['supplementary_article'],supplement)
    receipt.write_text(json.dumps({'url':plan['supplementary_article'],'sha256':hashlib.sha256(payload).hexdigest(),'retrieved_at_utc':datetime.now(timezone.utc).isoformat()},indent=2)+'\n')
else:payload=supplement.read_bytes()
receipt_data=json.loads(receipt.read_text());assert hashlib.sha256(payload).hexdigest()==receipt_data['sha256']
objects['sources/raw_documents'][supplement.name]=payload
objects['sources/manifest']['entries'].append({'path':supplement.name,**receipt_data,**article_metadata(payload)})
manifest=collect_sales_pdfs(html_manifest=objects['sources/manifest'],html_documents=objects['sources/raw_documents'],directory=raw,maximum_documents=plan['maximum_documents'],workers=plan['workers'])
events=pd.DataFrame(manifest['entries'])
for key in ['published_at_utc','created_at','modified_at']:
    events[key]=pd.to_datetime(events[key],utc=True)
events['clock_known']=events[['created_at','modified_at']].notna().all(axis=1)
events['assumed_anchor']=events[['published_at_utc','created_at','modified_at']].max(axis=1)
events['assumed_available_at']=events.assumed_anchor.dt.tz_convert('America/New_York').dt.tz_localize(None).dt.normalize().map(lambda d: calendar[calendar>d][0]+pd.Timedelta(hours=9,minutes=31) if (calendar>d).any() else pd.NaT)
events.loc[~events.clock_known,'assumed_available_at']=pd.NaT
events['observation_month']=pd.to_datetime(events.observation_month)
rows=[]
for day in calendar:
    known=events.loc[events.assumed_available_at.le(day+pd.Timedelta(hours=9,minutes=46))].sort_values(['observation_month','assumed_available_at'])
    row={'datetime':day,'instrument':'SIA_SALES','known':float(len(known)>0),'source_age_days':np.nan,**{key:np.nan for key in plan['fields']}}
    if len(known):
        e=known.iloc[-1];row.update({key:float(e[key]) for key in plan['fields']});row.update(source_age_days=(day-e.observation_month).days,observation_month=e.observation_month,assumed_available_at=e.assumed_available_at)
    rows.append(row)
state=pd.DataFrame(rows);fields=['known','source_age_days',*plan['fields']]
provider=root/plan['provider']
store=build_qlib_store_from_frame(state[['datetime','instrument',*fields]],provider_uri=provider,freq='day',as_of=plan['as_of'],field_map={k:k for k in fields},source='dated SIA GSR PDF tables; conditional attachment availability')
qlib.init(provider_uri=str(provider),region='us',kernels=1)
observed=D.features(['SIA_SALES'],['$'+k for k in fields],freq='day').xs('SIA_SALES',level='instrument')
assert observed.index.equals(calendar)
for k in fields:assert np.allclose(observed['$'+k],state[k],rtol=2e-7,atol=1e-6,equal_nan=True),k
audit={'expected_documents':manifest['expected_documents'],'parsed_documents':len(events),'failures':manifest['failures'],'observation_start':str(events.observation_month.min().date()),'observation_end':str(events.observation_month.max().date()),'duplicate_observation_months':int(events.observation_month.duplicated().sum()),'missing_observation_months':pd.period_range(events.observation_month.min(),events.observation_month.max(),freq='M').difference(pd.PeriodIndex(events.observation_month,freq='M')).astype(str).tolist(),'pdf_clock_unknown':int((~events.clock_known).sum()),'pdf_modified_after_article':int(events.modified_at.gt(events.published_at_utc).sum()),'native_calendar_rows':len(calendar),'known_days':int(state.known.sum()),'three_month_average_explicit_articles':int(events.article_three_month_average_explicit.sum()),'original_attachment_linkage_verified':False,'first_receipt_verified':False,'information_provider_is_trading_price':False}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='sia_gsr_pdf_source',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='sources',manifest=manifest,pdfs={p.name:p.read_bytes() for p in raw.glob('*.pdf')},parent_audit=parent_audit,supplementary_html=payload,supplementary_receipt=receipt_data,parser=(root/'src/data/sia_sales_pdf.py').read_text())
    rec.save_objects(artifact_path='analysis',events=events,information=state,audit=audit,store_manifest=store)
    rec.log_metrics(registered_artifact_count=8,parsed_documents=len(events))
summary={'experiment':plan['experiment'],'decision':'conditional_information_source_needs_vintage_proof' if not manifest['failures'] else 'needs_source_repair','source_audit':audit,'model_fits':0,'backtests':0,'objective_achieved':False,'goal_remains_active':True,'promotion_eligible':False,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(provider),'as_of':plan['as_of'],'region':'us','qlib_docs':plan['qlib_docs'],'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['sources/*','analysis/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts'
(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}))
events.drop(columns=['tables']).to_csv(card/'source_inventory.csv',index=False)
audit


{'expected_documents': 106,
 'parsed_documents': 106,
 'failures': [],
 'observation_start': '2017-11-01',
 'observation_end': '2026-08-01',
 'duplicate_observation_months': 0,
 'missing_observation_months': [],
 'pdf_clock_unknown': 6,
 'pdf_modified_after_article': 9,
 'native_calendar_rows': 2008,
 'known_days': 2008,
 'three_month_average_explicit_articles': 103,
 'original_attachment_linkage_verified': False,
 'first_receipt_verified': False,
 'information_provider_is_trading_price': False}

# SIA官方月度附件来源结论

已下载并解析106/106份官方GSR附件，覆盖2017-11至2026-08，观测月无重复。没有拟合模型或组合回测，稳定净SOXX超额目标尚未达成。

修复了57份表头换行导致的解析失败，以及1份2026-04报告使用“千美元”的单位差异。统一到十亿美元，按原文精度校验增长率、地区合计和三张表的当月总额，106份全部通过。同比、环比和地区同比离散度均来自各自同一版本，未以最新历史表回填早期比较值。初次失败Recorder与NAS归档保存在qualification_attempts.json。

文件时间方面，6份缺少可确认时区的时间，不用它们建立历史可用事件；9份PDF修改晚于公告。时间假设为公告、文件创建、文件修改三者最晚日期之后的下一个Qlib交易日09:31。未知事件不补时间；日历上保留上一条已知信息并记录年龄。2008个原生交易日均有已知历史信息，不代表每月都取得新信息。

已构建Qlib信息store并逐字段原生读回核对，记录source_age_days、known和3个销售特征。PDF时间元数据无法证明原始网页当时就链接当前附件，也不能认证真实首次接收；因此决定为conditional_information_source_needs_vintage_proof，只适用于明确假设的研究诊断，禁止直接晋升或声称PIT完全合格。

补齐官网“未分类”栏目下的2020-03公告附件后，106个月无缺失、无重复；该新增附件自身时间仍未知，因此只补全原始观测，不提前用于历史特征。两次来源修复前的Recorder保留。

下一步冻结6次线性信息增量检验：销售特征相对同样价格和信息年龄对照，严格标签成熟、完整保留2021/2022及复用后续窗口。先检验预测信息，失败不改符号/周期/阈值；通过也仍需要统一IBKR Tiered十万美元的完整账户、独立成本情景和新前向证据。当前生产SIM未改动。
